# 19 — Optimization: SGD, Momentum, Adam, and AdamW

**Master syllabus coverage:** V2 2.11–2.12

## Why this matters

Backpropagation supplies a gradient; the optimizer determines how that noisy local signal becomes a stable trajectory through parameter space.

## Learning objectives

- Implement gradient descent, momentum, and Adam updates.
- Observe learning-rate stability on an anisotropic objective.
- Verify a manual Adam step against PyTorch.
- Distinguish L2 regularization from decoupled weight decay.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Optimization turns gradients into parameter updates

Full-batch gradient descent uses
$\theta_{t+1}=\theta_t-\eta\nabla L(\theta_t)$. Stochastic gradient descent (SGD)
estimates the gradient from a minibatch. The learning rate $\eta$ controls the step,
not the final destination directly; too small is slow, too large can oscillate or diverge.


In [ ]:
import numpy as np
import torch

def objective(theta: np.ndarray) -> float:
    x, y = theta
    return 0.5 * (10 * x**2 + y**2)

def gradient(theta: np.ndarray) -> np.ndarray:
    x, y = theta
    return np.array([10 * x, y])

def run_gd(lr: float, steps: int = 30) -> list[float]:
    theta = np.array([3.0, 3.0])
    losses = [objective(theta)]
    for _ in range(steps):
        theta -= lr * gradient(theta)
        losses.append(objective(theta))
    return losses

for lr in (0.01, 0.1, 0.21):
    losses = run_gd(lr)
    print(f"lr={lr:.2f}: first={losses[0]:.2f}, final={losses[-1]:.4g}")


## 2. Momentum smooths and accelerates

One common convention:

$$v_t=\beta v_{t-1}+g_t,\qquad\theta_t=\theta_{t-1}-\eta v_t.$$

Momentum accumulates persistent directions and damps alternating ones. Be careful:
libraries use slightly different notation and update ordering.


In [ ]:
def optimize(kind: str, steps: int = 40, lr: float = 0.05) -> tuple[np.ndarray, list[float]]:
    theta = np.array([3.0, 3.0])
    velocity = np.zeros_like(theta)
    first = np.zeros_like(theta)
    second = np.zeros_like(theta)
    losses = []
    for step in range(1, steps + 1):
        g = gradient(theta)
        if kind == "sgd":
            update = g
        elif kind == "momentum":
            velocity = 0.9 * velocity + g
            update = velocity
        elif kind == "adam":
            first = 0.9 * first + 0.1 * g
            second = 0.999 * second + 0.001 * g**2
            first_hat = first / (1 - 0.9**step)
            second_hat = second / (1 - 0.999**step)
            update = first_hat / (np.sqrt(second_hat) + 1e-8)
        else:
            raise ValueError(kind)
        theta -= lr * update
        losses.append(objective(theta))
    return theta, losses

for kind in ("sgd", "momentum", "adam"):
    theta, losses = optimize(kind)
    print(f"{kind:8}: theta={theta.round(4)}, loss={losses[-1]:.5f}")


## 3. Adam adapts per-coordinate steps

Adam tracks exponential averages of gradients $m_t$ and squared gradients $v_t$,
corrects their early bias toward zero, and updates with

$$\theta_t=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.$$

Adaptive scaling is useful but does not remove the need to tune learning rate or inspect
gradient quality.


In [ ]:
# Match one manual Adam step with PyTorch.
parameter = torch.tensor([3.0, 3.0], dtype=torch.float64, requires_grad=True)
optimizer = torch.optim.Adam([parameter], lr=0.05, betas=(0.9, 0.999), eps=1e-8)
loss = 0.5 * (10 * parameter[0] ** 2 + parameter[1] ** 2)
loss.backward()
optimizer.step()

manual_theta, _ = optimize("adam", steps=1, lr=0.05)
torch.testing.assert_close(parameter.detach(), torch.tensor(manual_theta))
print("first Adam step:", parameter.detach())


## 4. AdamW decouples weight decay

L2 regularization adds $\lambda\theta$ to the loss gradient. AdamW instead decays
parameters separately from the adaptive gradient update. They are equivalent for plain
SGD under common conventions, but not generally for Adam because Adam rescales gradients.
Biases and normalization scale parameters are often excluded from decay.


In [ ]:
initial = torch.tensor([2.0], dtype=torch.float64)
adam_l2 = initial.clone().requires_grad_()
adamw = initial.clone().requires_grad_()
opt_l2 = torch.optim.Adam([adam_l2], lr=0.1, weight_decay=0.1)
opt_w = torch.optim.AdamW([adamw], lr=0.1, weight_decay=0.1)

# Same task gradient, different decay semantics.
for parameter_value, optimizer_value in ((adam_l2, opt_l2), (adamw, opt_w)):
    (0.5 * (parameter_value - 1) ** 2).backward()
    optimizer_value.step()
print("Adam + L2:", adam_l2.item(), "AdamW:", adamw.item())
assert not torch.allclose(adam_l2, adamw)


## Failure modes to recognize

- **Learning rate too high:** loss oscillates, grows, or becomes non-finite.
- **Missing bias correction:** Adam's earliest steps are systematically mis-scaled.
- **Epsilon misuse:** division becomes unstable or the adaptive effect is overwhelmed.
- **Universal weight decay:** biases and normalization parameters are regularized unintentionally.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Plot loss curves for all three optimizers under at least three learning rates.
2. Add stochastic gradient noise and compare trajectory smoothness.
3. Implement AdamW manually and match five PyTorch steps with fixed gradients.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can write each update equation, reproduce it in code, and diagnose a bad trajectory from loss and gradient evidence.

**Next:** 20 — Mathematical foundations capstone.
